In [1]:
!pip install thop

In [2]:
# Adicionem aqui as bibliotecas necessárias
import pandas as pd
import os
from PIL import Image
from torchvision import datasets, transforms, models
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import json
import csv
from thop import profile, clever_format
import timm

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
path = '/content/drive/MyDrive/Projeto LIPAI/Displastia'
dataset_name = "Displastia"

In [6]:
import json
import os

def save_metrics_json(base_dir,
                     arch_name, mode, aug,
                     seed, acc, f1_macro, f1_weighted):

    dir_path = f'{base_dir}/{dataset_name}/results'
    os.makedirs(dir_path, exist_ok=True)

    results = {
        "architecture": arch_name,
        "mode": mode,
        "aug": aug,
        "seed": seed,
        "accuracy": acc,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

    aug_str = "non_aug" if not aug else "aug"

    filename = f"{arch_name}_{mode}_{aug_str}_seed{seed}.json"
    path = os.path.join(dir_path, filename)

    with open(path, 'w') as f:
        json.dump(results, f, indent=4)

    print(f"Métricas salvas em: {path}")

# 1. Carregamento dos Dados

In [7]:
manifest_d = pd.read_csv(f'{path}/manifest.csv')
images_d = datasets.ImageFolder(root = f'{path}/dataset')

Divisão dos sets de acordo com o manifest

In [8]:
base_path = f'{path}/dataset'

def load_images_from_df(dataframe):
    images = []
    labels = []

    for idx, row in dataframe.iterrows():
        # Constrói o caminho completo: dataset/healthy/imagem.tif
        img_path = os.path.join(base_path, row['path'])

        # Leitura da imagem
        img = cv2.imread(img_path)
        if img is not None:
            images.append(img)
            labels.append(row['label_number'])
        else:
            print(f"Erro ao carregar: {img_path}")

    return np.array(images), np.array(labels)

# 3. Criar as divisões diretamente do CSV
train_df = manifest_d[manifest_d['sets'] == 'train']
val_df   = manifest_d[manifest_d['sets'] == 'val']
test_df  = manifest_d[manifest_d['sets'] == 'test']

# 4. Carregar os dados para as variáveis
print("Carregando imagens de treino...")
x_train, y_train = load_images_from_df(train_df)

print("Carregando imagens de validação...")
x_val, y_val = load_images_from_df(val_df)

print("Carregando imagens de teste...")
x_test, y_test = load_images_from_df(test_df)

# Conferindo os formatos
print("\nDivisão concluída:")
print(f"Treino: {x_train.shape}, {y_train.shape}")
print(f"Val:    {x_val.shape}, {y_val.shape}")
print(f"Teste:  {x_test.shape}, {y_test.shape}")

Carregando imagens de treino...
Carregando imagens de validação...
Carregando imagens de teste...

Divisão concluída:
Treino: (144, 250, 450, 3), (144,)
Val:    (38, 250, 450, 3), (38,)
Teste:  (46, 250, 450, 3), (46,)


## Transformações e Data Augmentation

Não queremos distorções brucas nas imagens histológicas, portanto, escolhemos aplicar apenas 3 operações geometricas.

* RandomHorizontalFlip: Espelhamento horizontal.
* RandomVerticalFlip: Espelhamento vertical.
* RandomRotation: Rotação suave de 15 graus.

In [9]:
transform_basic = transforms.Compose([ # Básica, sem augmentation
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

transform_augmentation = transforms.Compose([ # Com augmentation
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

## DataLoaders

Estou ultilizando uma classe auxiliar HistologyDataset, que envolve os arrays numpy, e os alinha com o pipeline do torchvision.transforms.


In [10]:
class HistologyDataset(Dataset):
    def __init__(self, images_array, labels_array, transform=None):
        self.images = images_array
        self.labels = labels_array
        self.transform = transform

    def __len__(self):
        # Tamanho total do dataset
        return len(self.labels)

    def __getitem__(self, idx):
        # Pega a imagem em formato Numpy
        img_np = self.images[idx]
        label = self.labels[idx]

        # Converte BGR para RGB
        img_rgb = cv2.cvtColor(img_np, cv2.COLOR_BGR2RGB)

        # Converte o array para um objeto Imagem (PIL)
        img_pil = Image.fromarray(img_rgb)

        # Aplica as transformações
        if self.transform:
            img_tensor = self.transform(img_pil)
        else:
            img_tensor = transforms.ToTensor()(img_pil)
        return img_tensor, label

In [11]:
BATCH_SIZE = 32

# Aplicação das transformações no treino.
train_dataset = HistologyDataset(x_train, y_train, transform=transform_augmentation)
val_dataset = HistologyDataset(x_val, y_val, transform=transform_basic)
test_dataset = HistologyDataset(x_test, y_test, transform=transform_basic)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders construídos!")
print(f"Total de lotes no treino: {len(train_loader)}")

DataLoaders construídos!
Total de lotes no treino: 5


# 2. Aplicação dos Algoritmos

Estou definindo uma função que permite aplicar os algoritmos de forma dinamica. Conforme as orientações, vamos usar:

* FS - From Scratch: Possui pesos aleatórios e aprende as caracteristicas visuais do zero.
* PT-ALL - Fine-tuning Completo: Possui pesos pré-treinados da ImageNet e será executado com todas as camadas treináveis.
* PT-FC - Backbone Congelado: Possui pesos pré-treinados da ImageNet e será executado com apenas o classificador treinável.

In [12]:
def build_model(name, mode, num_classes):
    """Fábrica de modelos. """

    if mode == 'FS':
        model = timm.create_model(name, pretrained=False, num_classes=num_classes)
    elif mode == 'PT-ALL':
        model = timm.create_model(name, pretrained=True, num_classes=num_classes)
    elif mode == 'PT-FC':
        model = timm.create_model(name, pretrained=True, num_classes=num_classes)
        # Congela os parâmetros
        for param in model.parameters():
            param.requires_grad = False
        # Descongela os parâmetros da última camada
        for param in model.get_classifier().parameters():
            param.requires_grad = True
    else:
        raise ValueError(f"Modo de treinamento {mode} não encontrado.")
    return model

# Como displasia é uma classificação binária, num_classes=2
modelo_teste = build_model('convnextv2_atto', 'PT-FC', num_classes=2)

total_params = sum(p.numel() for p in modelo_teste.parameters())
trainable_params = sum(p.numel() for p in modelo_teste.parameters() if p.requires_grad)

print(f"Total de parâmetros: {total_params:,}")
print(f"Parâmetros treináveis (PT-FC): {trainable_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/14.8M [00:00<?, ?B/s]

Total de parâmetros: 3,388,042
Parâmetros treináveis (PT-FC): 642


## 3. Treinamento

Da mesma forma que foi feito na célula anterior, estou criando uma função que contém a logica do laço de treinamento e avaliação. Dessa forma, a execução dos 72 experimentos vai ficar muito dinamica.

Como estamos treinando diversos modelos com parametros diferentes, decidimos acrescentar uma técnica de regularização chamada Early Stopping. A cada época, monitoramos uma métrica de validação, se essa métrica não melhorar por um número n consecutivos de épocas, o treinamento é interrompido automaticamente, e os pesos do melhor modelo são restaurados.

In [13]:
def train_and_evaluate(model, train_loader, val_loader, criterion, optimizer,
                       num_epochs=50, patience=10, device='cuda'):
    """ Função de treinamento e validação que retorna o melhor modelo e as métricas."""

    model.to(device)
    best_val_acc = 0.0
    best_model_state = None
    best_metrics = {}
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    # variaveis para o early stopping
    best_val_loss = float('inf')
    counter = 0

    for epoch in range(num_epochs):
        print(f"\n--- Epoch {epoch+1}/{num_epochs} ---")

        # Fase de Treinamento
        model.train()
        running_loss = 0.0
        all_train_preds, all_train_labels = [], []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad() # Zera o erro anterior
            outputs = model(inputs) # Previsão
            loss = criterion(outputs, labels) # Calcula o erro
            loss.backward() # Calcula os gradientes
            optimizer.step() # Ajusta os pesos

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            all_train_preds.extend(preds.cpu().numpy())
            all_train_labels.extend(labels.cpu().numpy())

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = accuracy_score(all_train_labels, all_train_preds)

        # Fase de Validação
        model.eval()
        val_loss = 0.0
        all_val_preds, all_val_labels = [], []

        # Desliga o cálculo dos gradientes
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                all_val_preds.extend(preds.cpu().numpy())
                all_val_labels.extend(labels.cpu().numpy())

        epoch_val_loss = val_loss / len(val_loader.dataset)
        epoch_val_acc = accuracy_score(all_val_labels, all_val_preds)

        # Salva o histórico
        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        # early stopping
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            counter = 0
            # Salva o melhor modelo
            best_model_state = model.state_dict().copy()
            best_metrics = {
                'epoch': epoch + 1,
                'val_acc': epoch_val_acc,
                'val_f1_macro': f1_score(all_val_labels, all_val_preds, average='macro'),
                'val_f1_weighted': f1_score(all_val_labels, all_val_preds, average='weighted')
            }
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping após {epoch+1} épocas. Melhor val_loss: {best_val_loss:.4f}")
                break

        print(f"Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.4f}")

    return best_model_state, best_metrics, history

## GFLOPs e Parâmetros da Arquitetura

Antes de rodar os experimentos, calculo os GFLOPs e o número de parâmetros de cada arquitetura, valores fixos para entrada 224×224 que vão para os gráficos comparativos.

In [14]:
def get_model_complexity(model_name, input_size=(1,3,224,224), device='cuda'):
    """Retorna GFLOPs, parâmetros para uma arquitetura."""

    # Cria modelo com pesos aleatórios
    model = timm.create_model(model_name, pretrained=False, num_classes=2)
    model.eval()
    model = model.to(device)
    input_tensor = torch.randn(*input_size).to(device)
    flops, params = profile(model, inputs=(input_tensor,), verbose=False)
    flops_gflops = flops / 1e9
    params_m = params / 1e6
    return flops_gflops, params_m

# Armazena a complexidade de cada arquitetura
complexidade = {}
arquiteturas = ['convnextv2_atto', 'convnextv2_pico']

print("Calculando GFLOPs e parâmetros para cada arquitetura...")

for arch in arquiteturas:
    gflops, params_m = get_model_complexity(arch)
    complexidade[arch] = {'gflops': gflops, 'params_m': params_m}
    print(f"{arch}: {params_m:.2f}M parâmetros, {gflops:.2f} GFLOPs")

Calculando GFLOPs e parâmetros para cada arquitetura...
convnextv2_atto: 3.37M parâmetros, 0.55 GFLOPs
convnextv2_pico: 8.52M parâmetros, 1.37 GFLOPs


## Execução do Loop de Treinamento

Chama as funções definidas acima para executar os experimentos, mudando os modos, condições de augmentation e seeds.

Além de salvar as métricas e historicos, consolido tudo em um arquivo CSV.

In [15]:
import os
import csv
import json

arquiteturas = ['convnextv2_atto', 'convnextv2_pico']
modos = ['FS', 'PT-FC', 'PT-ALL']
augmentations_list = [True, False]
seeds = [42, 123, 2025]

num_classes_dataset = len(np.unique(y_train))
criterion = nn.CrossEntropyLoss()

# Caminho para salvar os resultados
result_dir = f'{path}/{dataset_name}/results'
os.makedirs(result_dir, exist_ok=True)

# Arquivo CSV
csv_path = f'{result_dir}/resultados_consolidados.csv'
file_exists = os.path.isfile(csv_path)
with open(csv_path, mode='a', newline='') as f:
    writer = csv.writer(f)
    if not file_exists or os.stat(csv_path).st_size == 0:
        writer.writerow([
            'seed', 'dataset', 'model', 'training_mode', 'augmentation',
            'acc_test', 'f1_macro_test', 'f1_weighted_test',
            'num_params', 'gflops', 'best_epoch', 'val_acc_best'
        ])

print("Verificando experimentos concluídos e retomando...\n")

for arch in arquiteturas:
    # Complexidade calculada
    gflops = complexidade[arch]['gflops']
    params_m = complexidade[arch]['params_m']

    for modo in modos:
        for aug in augmentations_list:

            # Define transformação de treino conforme augmentation
            transformacao_atual = transform_augmentation if aug else transform_basic
            train_dataset_dinamico = HistologyDataset(x_train, y_train, transform=transformacao_atual)
            train_loader_dinamico = DataLoader(train_dataset_dinamico, batch_size=BATCH_SIZE, shuffle=True)

            for seed in seeds:

                # sistema de retomada do aprendizado
                aug_str = "aug" if aug else "non_aug"
                hist_filename = f"{arch}_{modo}_{aug_str}_seed{seed}_history.json"
                hist_path = os.path.join(result_dir, hist_filename)

                # Se o arquivo já existe no Drive, o modelo ja foi salvo
                if os.path.exists(hist_path):
                    print(f"Pulando: Arq={arch} | Modo={modo} | Aug={aug} | Seed={seed} (Já concluído)")
                    continue

                print(f"\n{'='*30}")
                print(f"Experimento: Arq={arch} | Modo={modo} | Aug={aug} | Seed={seed}")
                print(f"{'='*30}")

                # Fixa seeds
                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
                np.random.seed(seed)
                random.seed(seed)
                torch.backends.cudnn.deterministic = True

                # Constrói modelo
                modelo_atual = build_model(name=arch, mode=modo, num_classes=num_classes_dataset)

                # Otimizador
                optimizer = optim.Adam(modelo_atual.parameters(), lr=1e-4)

                # Treinamento
                best_state, best_metrics, history = train_and_evaluate(
                    model=modelo_atual,
                    train_loader=train_loader_dinamico,
                    val_loader=val_loader,
                    criterion=criterion,
                    optimizer=optimizer,
                    num_epochs=50,
                    device='cuda' if torch.cuda.is_available() else 'cpu'
                )

                # Carrega o melhor modelo
                modelo_atual.load_state_dict(best_state)

                # Avaliação
                modelo_atual.eval()
                all_test_preds = []
                all_test_labels = []

                with torch.no_grad():
                    for inputs, labels in test_loader:
                        inputs, labels = inputs.to(device), labels.to(device)
                        outputs = modelo_atual(inputs)
                        _, preds = torch.max(outputs, 1)
                        all_test_preds.extend(preds.cpu().numpy())
                        all_test_labels.extend(labels.cpu().numpy())

                acc_test = accuracy_score(all_test_labels, all_test_preds)
                f1_macro_test = f1_score(all_test_labels, all_test_preds, average='macro')
                f1_weighted_test = f1_score(all_test_labels, all_test_preds, average='weighted')

                # Salva curvas de aprendizado
                with open(hist_path, 'w') as f:
                    json.dump(history, f, indent=4)

                # Salva predições do teste
                pred_filename = f"{arch}_{modo}_{aug_str}_seed{seed}_predictions.json"
                pred_path = os.path.join(result_dir, pred_filename)
                with open(pred_path, 'w') as f:
                    json.dump({
                        # Converte em int pra poder salvar no json
                        'y_true': [int(v) for v in all_test_labels],
                        'y_pred': [int(v) for v in all_test_preds]
                    }, f, indent=4)

                # Escreve CSV
                with open(csv_path, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([
                        seed,
                        dataset_name,
                        arch,
                        modo,
                        'sim' if aug else 'nao',
                        acc_test,
                        f1_macro_test,
                        f1_weighted_test,
                        params_m,
                        gflops,
                        best_metrics['epoch'],
                        best_metrics['val_acc']
                    ])

                # Salva as métricas de teste no formato JSON
                save_metrics_json(
                    base_dir=path,
                    arch_name=arch,
                    mode=modo,
                    aug=aug,
                    seed=seed,
                    acc=acc_test,
                    f1_macro=f1_macro_test,
                    f1_weighted=f1_weighted_test
                )

print(f"\nExperimentos concluídos!")
print(f"CSV salvo/atualizado em: {csv_path}")
print(f"Históricos e predições salvos na pasta: {result_dir}")

Verificando experimentos concluídos e retomando...


Experimento: Arq=convnextv2_atto | Modo=FS | Aug=True | Seed=42

--- Epoch 1/50 ---
Train Loss: 0.7526 Acc: 0.5139 | Val Loss: 0.5178 Acc: 0.7368

--- Epoch 2/50 ---
Train Loss: 0.5831 Acc: 0.7014 | Val Loss: 0.4990 Acc: 0.7368

--- Epoch 3/50 ---
Train Loss: 0.5483 Acc: 0.6458 | Val Loss: 0.5353 Acc: 0.7105

--- Epoch 4/50 ---
Train Loss: 0.5215 Acc: 0.7569 | Val Loss: 0.4587 Acc: 0.8158

--- Epoch 5/50 ---
Train Loss: 0.5071 Acc: 0.7639 | Val Loss: 0.5135 Acc: 0.7105

--- Epoch 6/50 ---
Train Loss: 0.4796 Acc: 0.7569 | Val Loss: 0.4211 Acc: 0.8421

--- Epoch 7/50 ---
Train Loss: 0.4591 Acc: 0.8264 | Val Loss: 0.3941 Acc: 0.8158

--- Epoch 8/50 ---
Train Loss: 0.4113 Acc: 0.8125 | Val Loss: 0.4158 Acc: 0.7895

--- Epoch 9/50 ---
Train Loss: 0.4060 Acc: 0.8125 | Val Loss: 0.3677 Acc: 0.8947

--- Epoch 10/50 ---
Train Loss: 0.3714 Acc: 0.8542 | Val Loss: 0.5751 Acc: 0.7105

--- Epoch 11/50 ---
Train Loss: 0.4360 Acc: 0.8194 | Val Loss

model.safetensors:   0%|          | 0.00/36.3M [00:00<?, ?B/s]


--- Epoch 1/50 ---
Train Loss: 0.6153 Acc: 0.6389 | Val Loss: 0.6295 Acc: 0.6053

--- Epoch 2/50 ---
Train Loss: 0.5322 Acc: 0.7222 | Val Loss: 0.6560 Acc: 0.6579

--- Epoch 3/50 ---
Train Loss: 0.4936 Acc: 0.7778 | Val Loss: 0.6624 Acc: 0.6842

--- Epoch 4/50 ---
Train Loss: 0.5129 Acc: 0.7431 | Val Loss: 0.6553 Acc: 0.6842

--- Epoch 5/50 ---
Train Loss: 0.4596 Acc: 0.7917 | Val Loss: 0.6355 Acc: 0.6842

--- Epoch 6/50 ---
Train Loss: 0.4157 Acc: 0.7986 | Val Loss: 0.5990 Acc: 0.7105

--- Epoch 7/50 ---
Train Loss: 0.3930 Acc: 0.8194 | Val Loss: 0.5675 Acc: 0.7368

--- Epoch 8/50 ---
Train Loss: 0.3800 Acc: 0.8403 | Val Loss: 0.5518 Acc: 0.7368

--- Epoch 9/50 ---
Train Loss: 0.3824 Acc: 0.8542 | Val Loss: 0.5321 Acc: 0.7368

--- Epoch 10/50 ---
Train Loss: 0.3874 Acc: 0.8333 | Val Loss: 0.5114 Acc: 0.7895

--- Epoch 11/50 ---
Train Loss: 0.3382 Acc: 0.8819 | Val Loss: 0.4884 Acc: 0.7895

--- Epoch 12/50 ---
Train Loss: 0.3394 Acc: 0.8681 | Val Loss: 0.4737 Acc: 0.7895

--- Epoch 13

# 4. Comparativos Globais



In [ ]:
import json
import os
import pandas as pd

df_displasia = pd.read_csv("/content/resultados_consolidados.csv")

df_displasia

**Exploration**

In [ ]:
df_displasia.head()

In [ ]:
df_displasia.shape #Linhas / Coluna

In [ ]:
df_displasia.info()

In [ ]:
df_displasia.describe()

In [ ]:
df_displasia["model"].unique() #Quais modelos existem

In [ ]:
df_displasia["training_mode"].unique() #Quais modos existem

In [ ]:
df_displasia["seed"].unique() #Quais Seeds

In [ ]:
df_displasia["augmentation"].value_counts() #Balanceamento Augmentation

In [ ]:
#Melhor Resultado
df_displasia.sort_values("acc_test", ascending=False)

In [ ]:
#Pior Resultado
df_displasia.sort_values("acc_test", ascending=True)

# Valores Medios

In [ ]:
df_displasia.groupby("training_mode")["acc_test"].mean() #Media por modo

In [ ]:
df_displasia.groupby("model")["acc_test"].mean() #Media por Modelo

In [ ]:
df_displasia.groupby("augmentation")["acc_test"].mean() #Augmentation foi relevante?

# Relatorio 1 - Análise Inicial dos Dados

O DataFrame consolidado referente ao dataset Displasia apresentou um total de 37 experimentos distribuídos em 12 atributos distintos. Durante a análise exploratória dos dados, não foram identificados valores ausentes ou inconsistências, indicando que os resultados foram armazenados corretamente e que o conjunto encontra-se íntegro para análise estatística e comparativa.

Os experimentos foram realizados utilizando duas arquiteturas de redes neurais convolucionais:

* ConvNeXtV2 Atto
* ConvNeXtV2 Pico

Além disso, foram avaliados três modos distintos de treinamento:

* FS (From Scratch)
* PT-FC (Pré-treinado com Backbone Congelado)
* PT-ALL (Pré-treinado com Fine-Tuning Completo)

Cada configuração experimental foi executada utilizando três seeds diferentes (42, 123 e 2025), com o objetivo de reduzir possíveis influências da aleatoriedade no processo de treinamento e permitir uma avaliação mais confiável da estabilidade dos modelos.

**Data Augmentation**

A variável augmentation representa a utilização ou não de técnicas de aumento artificial de dados durante o treinamento do modelo.

Nos experimentos:

* sim → indica que técnicas de Data Augmentation foram utilizadas;
* não → indica que o treinamento ocorreu utilizando apenas as imagens originais.

As técnicas de Data Augmentation têm como objetivo aumentar artificialmente a variabilidade do conjunto de treinamento por meio de pequenas transformações nas imagens, como rotações, espelhamentos e redimensionamentos. Isso permite que o modelo seja exposto a diferentes variações dos mesmos padrões visuais, reduzindo o risco de overfitting e aumentando a capacidade de generalização.

Durante a análise dos experimentos, observou-se uma distribuição praticamente equilibrada entre execuções com e sem augmentation:

* Sem augmentation: 19 experimentos
* Com augmentation: 18 experimentos

Essa distribuição é considerada quase balanceada porque existe uma quantidade muito semelhante de experimentos nos dois cenários. Isso é importante para evitar comparações enviesadas, garantindo que ambos os grupos possuam representatividade semelhante durante a análise estatística.

**Resultados Augmentation**

Os resultados mostraram que os experimentos com Data Augmentation apresentaram desempenho médio superior aos experimentos sem augmentation.

As médias observadas foram:

* Sem augmentation: 0.875
* Com augmentation: 0.938

Essa diferença indica que o uso de augmentation contribuiu positivamente para o desempenho dos modelos no dataset Displasia.

Esse comportamento pode ser explicado pelo fato de que o augmentation aumenta artificialmente a diversidade das imagens de treinamento, permitindo que a rede neural aprenda padrões mais generalizáveis e menos dependentes de características específicas do conjunto original.

Além disso, técnicas de augmentation ajudam a reduzir problemas de overfitting, especialmente em datasets relativamente pequenos, como frequentemente ocorre em aplicações de imagens médicas e histológicas.

**Comparação entre as Arquiteturas**

Durante a análise dos resultados obtidos no dataset Displasia, observou-se uma diferença significativa de desempenho entre as arquiteturas ConvNeXtV2 Atto e ConvNeXtV2 Pico.

As médias de acurácia observadas foram:

* ConvNeXtV2 Atto: 0.946
* ConvNeXtV2 Pico: 0.867

Os resultados indicam que a arquitetura ConvNeXtV2 Atto apresentou desempenho superior na tarefa de classificação histológica quando comparada à ConvNeXtV2 Pico.

De forma simplificada, isso significa que a arquitetura Atto conseguiu identificar e diferenciar melhor os padrões presentes nas imagens do dataset. Em problemas de visão computacional médica, pequenas diferenças visuais entre classes podem ser extremamente importantes, e arquiteturas mais adequadas conseguem extrair essas características de maneira mais eficiente.

Além disso, os resultados sugerem que a arquitetura Pico pode ter apresentado menor capacidade de generalização para este problema específico. Embora ambas pertençam à mesma família de modelos, pequenas diferenças estruturais podem impactar diretamente a capacidade de aprendizado da rede neural.

**Comparação entre os modos**

A análise dos diferentes modos de treinamento apresentou resultados relevantes para a interpretação do comportamento dos modelos no dataset Displasia.

As médias de acurácia observadas foram:

* FS (From Scratch): 0.918
* PT-FC (Pré-treinado com Backbone Congelado): 0.927
* PT-ALL (Fine-Tuning Completo): 0.874

Os resultados mostram que o modo PT-FC apresentou o melhor desempenho médio entre os três modos avaliados.

Inicialmente, esperava-se que o modo PT-ALL apresentasse os melhores resultados, uma vez que ele permite o ajuste completo de todas as camadas do modelo. Entretanto, os experimentos demonstraram um comportamento diferente do esperado.

Uma possível explicação para esse resultado é a ocorrência de overfitting durante o fine-tuning completo. Nesse cenário, o modelo passa a se adaptar excessivamente aos dados de treinamento, reduzindo sua capacidade de generalização para imagens não vistas anteriormente.

Esse comportamento é relativamente comum em aplicações médicas e histológicas, especialmente quando os datasets possuem tamanho limitado. Como o modo PT-ALL ajusta todas as camadas da rede neural simultaneamente, o modelo pode acabar aprendendo detalhes muito específicos do conjunto de treino, em vez de aprender padrões mais generalizáveis.

Além disso, alguns experimentos apresentaram resultados consideravelmente inferiores, como acurácias próximas de 0.500 e 0.695, principalmente nas configurações envolvendo ConvNeXtV2 Pico com PT-ALL sem augmentation. Esses resultados indicam possíveis problemas de estabilidade durante o treinamento, além de sugerirem dificuldades de convergência do modelo nesse cenário específico.

**Características Gerais do Dataset**

Os resultados obtidos sugerem que o dataset Displasia apresenta uma tarefa de classificação relativamente bem definida para algumas configurações experimentais.

Diversos experimentos alcançaram acurácia igual a 1.000, indicando que o modelo conseguiu classificar corretamente todas as imagens do conjunto de teste em determinadas execuções.

Esse comportamento pode indicar:

* boa separação visual entre as classes;
* presença de padrões histológicos relativamente distintos;
* menor complexidade da tarefa binária em comparação com problemas multiclasse.

Entretanto, também foram identificados resultados significativamente inferiores em algumas execuções específicas, caracterizando a presença de outliers experimentais.

Os valores próximos de 0.500 e 0.695 merecem atenção especial, pois impactam negativamente as médias gerais, principalmente no modo PT-ALL. Essas variações sugerem que determinadas combinações de arquitetura, modo de treinamento e augmentation podem apresentar maior instabilidade durante o processo de aprendizado.

**Análise de Complexidade Computacional**

A análise dos parâmetros e da complexidade computacional dos modelos também apresentou resultados interessantes.

Observou-se que a arquitetura ConvNeXtV2 Atto apresentou menor quantidade de parâmetros e menor custo computacional em GFLOPs quando comparada à ConvNeXtV2 Pico.

Mesmo sendo uma arquitetura mais leve, a ConvNeXtV2 Atto obteve desempenho superior nos experimentos realizados.

Esse resultado é relevante, pois demonstra que modelos menores nem sempre apresentam desempenho inferior. Em alguns cenários, arquiteturas mais compactas podem alcançar melhores resultados devido à maior estabilidade de treinamento e menor tendência ao overfitting.

Além disso, modelos mais leves apresentam vantagens práticas importantes, como menor custo computacional, menor consumo de memória e maior facilidade de implantação em ambientes com recursos limitados.

# Etapa 2 - Graficos




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs("/content/figuras_displasia", exist_ok=True)

In [ ]:
#Tabela com média e desvio padrão
tabela_resumo = (
    df_displasia
    .groupby(["model", "training_mode", "augmentation"])
    .agg(
        acc_mean=("acc_test", "mean"),
        acc_std=("acc_test", "std"),
        f1_macro_mean=("f1_macro_test", "mean"),
        f1_macro_std=("f1_macro_test", "std"),
        f1_weighted_mean=("f1_weighted_test", "mean"),
        f1_weighted_std=("f1_weighted_test", "std")
    )
    .reset_index()
)

tabela_resumo

In [ ]:
tabela_resumo.to_csv("/content/figuras_displasia/tabela_media_desvio_displasia.csv", index=False)

In [ ]:
#Gráfico de barras do F1-score
grafico_f1 = tabela_resumo.copy()

grafico_f1["config"] = (
    grafico_f1["model"] + " | " +
    grafico_f1["training_mode"] + " | aug=" +
    grafico_f1["augmentation"].astype(str)
)

plt.figure(figsize=(14, 6))
plt.bar(grafico_f1["config"], grafico_f1["f1_macro_mean"], yerr=grafico_f1["f1_macro_std"], capsize=5)
plt.xticks(rotation=90)
plt.ylabel("F1-score macro médio")
plt.xlabel("Configuração experimental")
plt.title("Displasia - F1-score macro por arquitetura, modo de treinamento e augmentation")
plt.tight_layout()
plt.savefig("/content/figuras_displasia/f1_score_global_displasia.pdf")
plt.show()

In [ ]:
#Gráfico de número de parâmetros por arquitetura
params_por_modelo = (
    df_displasia
    .groupby("model")["num_params"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.bar(params_por_modelo["model"], params_por_modelo["num_params"])
plt.ylabel("Número de parâmetros")
plt.xlabel("Arquitetura")
plt.title("Displasia - Número de parâmetros por arquitetura")
plt.tight_layout()
plt.savefig("/content/figuras_displasia/parametros_por_arquitetura_displasia.pdf")
plt.show()

In [ ]:
#Gráfico de GFLOPs por arquitetura
gflops_por_modelo = (
    df_displasia
    .groupby("model")["gflops"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.bar(gflops_por_modelo["model"], gflops_por_modelo["gflops"])
plt.ylabel("GFLOPs")
plt.xlabel("Arquitetura")
plt.title("Displasia - Complexidade computacional por arquitetura")
plt.tight_layout()
plt.savefig("/content/figuras_displasia/gflops_por_arquitetura_displasia.pdf")
plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt

with open("/content/convnextv2_atto_PT-ALL_aug_seed42_history.json", "r") as f:
    history = json.load(f)

epochs = range(1, len(history["train_loss"]) + 1)

# LOSS
plt.figure(figsize=(8,5))

plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")

plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.title("Curva de Loss")
plt.legend()

plt.tight_layout()
plt.savefig("/content/curva_loss.pdf")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(epochs, history["train_acc"], label="Train Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")

plt.xlabel("Épocas")
plt.ylabel("Accuracy")
plt.title("Curva de Accuracy")
plt.legend()

plt.tight_layout()
plt.savefig("/content/curva_accuracy.pdf")

plt.show()

In [ ]:
import json
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

with open("/content/convnextv2_atto_PT-ALL_aug_seed42_predictions.json", "r") as f:
    preds = json.load(f)

y_true = preds["y_true"]
y_pred = preds["y_pred"]

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)

fig, ax = plt.subplots(figsize=(6,6))

disp.plot(ax=ax)

plt.title("Matriz de Confusão")

plt.tight_layout()
plt.savefig("/content/matriz_confusao.pdf")

plt.show()

In [ ]:
import matplotlib.pyplot as plt

df_displasia.boxplot(column="acc_test", by="training_mode", figsize=(8,5))

plt.title("Distribuição da Accuracy por Modo")
plt.suptitle("")
plt.ylabel("Accuracy")

plt.show()

In [ ]:
tabela_final = (
    df_displasia
    .groupby(["model", "training_mode", "augmentation"])
    .agg({
        "acc_test": ["mean", "std"],
        "f1_macro_test": ["mean", "std"],
        "f1_weighted_test": ["mean", "std"]
    })
)

tabela_final

In [ ]:
tabela_formatada = tabela_final.copy()

for metrica in ["acc_test", "f1_macro_test", "f1_weighted_test"]:

    tabela_formatada[(metrica, "resultado")] = (
        tabela_formatada[(metrica, "mean")].round(3).astype(str)
        + " ± " +
        tabela_formatada[(metrica, "std")].round(3).astype(str)
    )

tabela_formatada = tabela_formatada[[
    ("acc_test", "resultado"),
    ("f1_macro_test", "resultado"),
    ("f1_weighted_test", "resultado")
]]

tabela_formatada.columns = [
    "Accuracy",
    "F1 Macro",
    "F1 Weighted"
]

tabela_formatada

In [ ]:
tabela_formatada.to_csv(
    "/content/tabela_media_desvio_formatada.csv"
)

CURVAS/MATRIZES

In [ ]:
melhor = df_displasia.sort_values(
    "acc_test",
    ascending=False
)

melhor.head(3)



In [ ]:
pior = df_displasia.sort_values(
    "acc_test",
    ascending=True
)

pior.head(3)



In [ ]:
mediana = df_displasia.iloc[
    len(df_displasia)//2
]

mediana

# Relatorio 2 - Discussão Final

Os resultados obtidos no dataset Displasia demonstraram que as diferentes configurações experimentais impactaram significativamente o desempenho dos modelos avaliados.

De maneira geral, observou-se que o uso de Data Augmentation contribuiu positivamente para a capacidade de generalização das redes neurais, resultando em melhorias consistentes nas métricas de avaliação. Esse comportamento indica que o aumento artificial da variabilidade das imagens auxiliou os modelos na extração de padrões mais robustos e menos dependentes de características específicas do conjunto de treinamento.

Entre as arquiteturas avaliadas, a ConvNeXtV2 Atto apresentou os melhores resultados médios de desempenho, superando a ConvNeXtV2 Pico em praticamente todas as configurações analisadas. Além disso, a arquitetura Atto também apresentou menor custo computacional em termos de parâmetros e GFLOPs, demonstrando uma relação favorável entre desempenho e eficiência computacional.

Os experimentos também mostraram diferenças importantes entre os modos de treinamento. O modo PT-FC apresentou maior estabilidade média em comparação ao PT-ALL, comportamento que pode estar relacionado à ocorrência de overfitting durante o fine-tuning completo. Esse cenário é relativamente comum em aplicações de visão computacional médica, especialmente quando os datasets possuem tamanho reduzido.

Outro ponto relevante foi a presença de execuções com desempenho significativamente inferior em determinadas configurações experimentais, principalmente envolvendo a arquitetura ConvNeXtV2 Pico no modo PT-ALL sem augmentation. Esses resultados sugerem possíveis dificuldades de convergência e maior sensibilidade às variações aleatórias do treinamento.

De forma geral, os resultados obtidos indicam que o dataset Displasia apresentou boa separação entre as classes avaliadas, uma vez que diversas execuções alcançaram métricas próximas de 1.000. Ainda assim, a análise das curvas de aprendizado e das matrizes de confusão demonstrou que determinadas configurações permanecem suscetíveis à instabilidade e ao overfitting.

Por fim, os experimentos evidenciam a importância da escolha adequada da arquitetura, do modo de treinamento e do uso de técnicas de augmentation em problemas de classificação histológica, mostrando que modelos mais leves e estáveis podem alcançar desempenho superior mesmo com menor custo computacional.